In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations

# =====================================================
# LOAD YOUTUBE GRAPH DATA
# =====================================================

df = pd.read_csv(
    "../data_preprocess/processed_data/graph_data/youtube_graph.csv"
)

print("=" * 60)
print("YOUTUBE GRAPH DATA")
print("=" * 60)

print(df.shape)

In [ ]:
# CREATE GRAPH
# =====================================================

G = nx.Graph()

# =====================================================
# ADD NODES
# =====================================================

for idx, row in df.iterrows():

    G.add_node(

        row["post_id"],

        topic=row["topic"],
        media_type=row["media_type"],
        location=row["location"],
        followers=row["followers"],
        verified=row["verified"],
        popularity=row["popularity"]
    )

print("\nNodes added:", G.number_of_nodes())

# =====================================================
# FEATURE-GUIDED EDGE CONSTRUCTION
# =====================================================
# Based on YouTube feature importance
# =====================================================

edge_count = 0

# =====================================================
# GROUP 1
# TOPIC + MEDIA TYPE
# =====================================================

group_1 = df.groupby(
    ["topic", "media_type"]
)

for _, group in group_1:

    post_ids = group["post_id"].tolist()

    # connect inside group
    for p1, p2 in combinations(post_ids, 2):

        if not G.has_edge(p1, p2):

            G.add_edge(

                p1,
                p2,

                relation="topic_media",
                weight=2
            )

            edge_count += 1

# =====================================================
# GROUP 2
# LOCATION + TOPIC
# =====================================================

group_2 = df.groupby(
    ["location", "topic"]
)

for _, group in group_2:

    post_ids = group["post_id"].tolist()

    for p1, p2 in combinations(post_ids, 2):

        if not G.has_edge(p1, p2):

            G.add_edge(

                p1,
                p2,

                relation="location_topic",
                weight=2
            )

            edge_count += 1

# =====================================================
# GROUP 3
# VERIFIED USERS
# =====================================================

verified_group = df[
    df["verified"] == 1
]

verified_posts = verified_group[
    "post_id"
].tolist()

for p1, p2 in combinations(
    verified_posts,
    2
):

    if not G.has_edge(p1, p2):

        G.add_edge(

            p1,
            p2,

            relation="verified_creator",
            weight=3
        )

        edge_count += 1

print("\nEdges created:", edge_count)

# =====================================================
# GRAPH INFORMATION
# =====================================================

print("\n" + "=" * 60)
print("GRAPH INFORMATION")
print("=" * 60)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())